# Privacy in Machine Learning: Hands-On Lab
## Attacking a Model with Membership Inference Attack & Defending it with Differential Privacy

**Instructions**: This notebook contains `# TODO` sections where you need to fill in code.
Each TODO requires only 1-3 lines of code. Validation cells will check your work.

**Time**: ~60 minutes total

1: Set up the environment

- **Goal:** Prepares the notebook dependencies, compute device, and reproducible random state.

- **What runs:** Installs the required packages; imports PyTorch, MedMNIST, Opacus, scikit-learn, NumPy, and Matplotlib; selects CUDA, MPS, or CPU; and sets random seeds.

- **Outcome:** The environment reports the selected device and confirms that all packages were imported successfully.

- **TODO:** N/A


In [1]:
!pip install torch torchvision medmnist opacus scikit-learn matplotlib numpy --quiet

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from opacus import PrivacyEngine          # handles DP-SGD for us in Phase 3
import medmnist
from medmnist import PneumoniaMNIST
import warnings
warnings.filterwarnings('ignore')

# Prefer GPU (CUDA), then Apple Silicon (MPS), then CPU as a fallback.
if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

# Fix seeds so every student gets reproducible, comparable numbers.
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("All packages imported successfully!")

## Phase 0: Warm-up and Orientation (~5 min)
**Phase 0 Goal**: Confirm environment works, explore the dataset

2: Load and inspect PneumoniaMNIST

- **Goal:** Loads the training and test chest X-ray datasets, normalizes the images, and examines class balance.

- **What runs:** Uses `transforms.Compose`, `PneumoniaMNIST`, `np.unique`, and Matplotlib to prepare the data and plot Normal versus Pneumonia counts.

- **Outcome:** The dataset sizes, tensor shapes, and training-class distribution are displayed.

- **TODO:** N/A

**About PneumoniaMNIST:**

- The PneumoniaMNIST is based on a prior dataset of 5,856 pediatric chest X-Ray images. 

- The task is binary-class classification of pneumonia against normal.

- The source images are gray-scale, and their sizes are (384−2,916)×(127−2,713). We center-crop the images and resize them into 1×28×28.

- label information: {'0': 'normal', '1': 'pneumonia'}



In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

train_dataset = PneumoniaMNIST(split='train', transform=transform, download=True)
test_dataset = PneumoniaMNIST(split='test', transform=transform, download=True)

print(f"Training set size: {len(train_dataset)}")
print(f"Test set size: {len(test_dataset)}")
print(f"Image shape: {train_dataset[0][0].shape}")
print(f"Label shape: {train_dataset[0][1].shape}")

# Class imbalance matters: a skewed dataset can inflate accuracy,
# so we visualize how many Normal vs Pneumonia examples we have.
train_labels = np.array([train_dataset[i][1].item() for i in range(len(train_dataset))])
unique, counts = np.unique(train_labels, return_counts=True)

fig, ax = plt.subplots(figsize=(6, 4))
class_names = ['Normal', 'Pneumonia']
ax.bar(class_names, counts, color=['steelblue', 'coral'])
ax.set_ylabel('Count')
ax.set_title('Training Set Class Distribution')
for i, (name, count) in enumerate(zip(class_names, counts)):
    ax.text(i, count + 20, str(count), ha='center', fontsize=12)
plt.tight_layout()
plt.show()

3: Preview sample X-rays

- **Goal:** Provides a visual sanity check of the normalized PneumoniaMNIST images and labels.

- **What runs:** Uses `plt.subplots`, iterates through the first eight training samples, reverses normalization for display, and renders each image with `imshow`.

- **Outcome:** A 2-by-4 gallery of labeled Normal and Pneumonia X-rays is shown.

- **TODO:** N/A


In [3]:
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    # Undo the Normalize step so the X-ray displays with natural contrast.
    img_display = img.squeeze().numpy() * 0.5 + 0.5
    ax.imshow(img_display, cmap='gray')
    ax.set_title(f"{'Pneumonia' if label.item() == 1 else 'Normal'}")
    ax.axis('off')
plt.suptitle('Sample Images from PneumoniaMNIST', fontsize=14)
plt.tight_layout()
plt.show()

4: Create membership-inference splits

- **Goal:** Divides the training pool into disjoint member, non-member, and validation subsets for the privacy experiment.

- **What runs:** Uses `np.random.permutation`, `Subset`, and `DataLoader` to create a 40% member split, a 40% non-member split, and a 20% validation split.

- **Outcome:** Four loaders are created, and the size of each dataset split is printed.

- **TODO:** N/A


In [4]:
n_train = len(train_dataset)
indices = np.random.permutation(n_train)   # shuffle so the split is unbiased

n_members = int(0.4 * n_train)
n_nonmembers = int(0.4 * n_train)
n_val = n_train - n_members - n_nonmembers

member_indices = indices[:n_members]
nonmember_indices = indices[n_members:n_members + n_nonmembers]
val_indices = indices[n_members + n_nonmembers:]

member_dataset = Subset(train_dataset, member_indices)
nonmember_dataset = Subset(train_dataset, nonmember_indices)
val_dataset = Subset(train_dataset, val_indices)

# shuffle=True only for the set we train on; evaluation loaders stay ordered.
member_loader = DataLoader(member_dataset, batch_size=64, shuffle=True)
nonmember_loader = DataLoader(nonmember_dataset, batch_size=64, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Members (training data): {len(member_dataset)}")
print(f"Non-members (held out): {len(nonmember_dataset)}")
print(f"Validation: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")

5: Validate the data setup

- **Goal:** Checks that Phase 0 produced usable member and non-member datasets and selected a compute device.

- **What runs:** Runs three conditional checks with `len(...)` and the `device` value, then counts and prints passing checks.

- **Outcome:** A Phase 0 validation summary reports up to three passes and confirms completion when all checks succeed.

- **TODO:** N/A


In [5]:
print("=" * 50)
print("PHASE 0 VALIDATION")
print("=" * 50)
checks = 0
if len(member_dataset) > 0:
    print(f"[PASS] Member dataset loaded: {len(member_dataset)} samples")
    checks += 1
if len(nonmember_dataset) > 0:
    print(f"[PASS] Non-member dataset loaded: {len(nonmember_dataset)} samples")
    checks += 1
if device is not None:
    print(f"[PASS] Device set: {device}")
    checks += 1
print(f"\nResult: {checks}/3 checks passed.")
if checks == 3:
    print("Phase 0 COMPLETE!")
print("=" * 50)

6: Define the baseline model

- **Goal:** Defines the fully connected neural network used for binary PneumoniaMNIST classification.

- **What runs:** Creates the `SimpleFC` class and `create_model()` helper, moves a test model to the selected device, and counts its parameters.

- **Outcome:** The model architecture becomes available for later training, and its total parameter count is printed.

- **TODO:** N/A


In [ ]:
class SimpleFC(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 1)   # single logit -> Normal vs Pneumonia
    def forward(self, x):
        x = x.view(x.size(0), -1)      # flatten the 28x28 image into a vector
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

def create_model():
    return SimpleFC().to(device)

test_model = create_model()
total_params = sum(p.numel() for p in test_model.parameters())
print(f"Total parameters: {total_params:,}")
del test_model

## Phase 1: Seeing the Leak — Overfitting as the Root Cause (~13 min)
**Phase 1 Goal**: Train a baseline model and observe the generalization gap
**Target**: Train accuracy > 95%, Test accuracy ~85-93%, Gap > 3%

7: Define training and evaluation helpers

- **Goal:** Introduces reusable functions for one training epoch and for classification-accuracy evaluation.

- **What runs:** `train_one_epoch` performs optimization over a data loader, while `evaluate_accuracy` uses `torch.no_grad`, sigmoid thresholding, and label comparisons.

- **Outcome:** Once completed, the helpers will return average training loss and percentage accuracy for later phases.

- **TODO:** Complete TODO 1 by adding the forward pass, loss calculation, backpropagation, and optimizer step in `train_one_epoch`.


In [7]:
def train_one_epoch(model, dataloader, optimizer, criterion):
    """Train for one epoch, return average loss."""
    model.train()
    total_loss = 0.0
    n_batches = 0
    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.float().to(device).view(-1, 1)   # shape (B,1) for BCE

        # TODO 1: Complete the training step (3 lines)
        # Hint: zero gradients, compute loss from model outputs, backprop and step
        optimizer.zero_grad()
        outputs = None  # <-- Replace None with forward pass
        loss = None     # <-- Replace None with loss computation
        # <-- Add backward and step calls

        total_loss += loss.item()
        n_batches += 1
    return total_loss / n_batches

def evaluate_accuracy(model, dataloader):
    """Compute accuracy as percentage."""
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():   # no gradients needed at eval -> faster, less memory
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.float().to(device).view(-1, 1)
            outputs = model(images)
            # logit >= 0  <=>  sigmoid(logit) >= 0.5  -> predict class 1.
            preds = (torch.sigmoid(outputs) >= 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total

print("Training functions defined.")

8: Train the non-private baseline

- **Goal:** Trains a baseline model on member data and measures its generalization gap against unseen test data.

- **What runs:** Creates a model with `create_model`, optimizes it with Adam and `BCEWithLogitsLoss`, calls `train_one_epoch` and `evaluate_accuracy` for 20 epochs, and plots both accuracy curves.

- **Outcome:** Training progress, final train/test accuracy, and a visualization of the generalization gap are produced.

- **TODO:** N/A


In [ ]:
model = create_model()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()   # combines sigmoid + binary cross-entropy

NUM_EPOCHS = 20
train_accs = []
test_accs = []

print(f"Training baseline model for {NUM_EPOCHS} epochs...")
for epoch in range(NUM_EPOCHS):
    loss = train_one_epoch(model, member_loader, optimizer, criterion)
    train_acc = evaluate_accuracy(model, member_loader)   # accuracy on seen data
    test_acc = evaluate_accuracy(model, test_loader)      # accuracy on unseen data
    train_accs.append(train_acc)
    test_accs.append(test_acc)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}: Loss={loss:.4f}, Train={train_acc:.1f}%, Test={test_acc:.1f}%")

# Plot the two curves; the shaded band highlights the generalization gap.
plt.figure(figsize=(8, 5))
plt.plot(range(1, NUM_EPOCHS+1), train_accs, 'b-o', label='Train (members)', markersize=4)
plt.plot(range(1, NUM_EPOCHS+1), test_accs, 'r-o', label='Test', markersize=4)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.title('Baseline Model: Train vs Test Accuracy', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
gap_final = train_accs[-1] - test_accs[-1]
plt.axhspan(test_accs[-1], train_accs[-1], alpha=0.1, color='red', label=f'Gap={gap_final:.1f}%')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"\nFinal: Train={train_accs[-1]:.1f}%, Test={test_accs[-1]:.1f}%, Gap={gap_final:.1f}%")

9: Validate baseline leakage

- **Goal:** Checks whether the baseline reached high member accuracy, reasonable test accuracy, and a measurable generalization gap.

- **What runs:** Reads the final accuracy values, applies three threshold checks, and prints pass or fail messages.

- **Outcome:** The Phase 1 summary indicates whether the baseline behavior is suitable for demonstrating membership leakage.

- **TODO:** N/A


In [9]:
print("=" * 50)
print("PHASE 1 VALIDATION")
print("=" * 50)
checks_passed = 0
total_checks = 3

final_train_acc = train_accs[-1]
final_test_acc = test_accs[-1]
gap = final_train_acc - final_test_acc

if final_train_acc > 95:
    print(f"[PASS] Train accuracy = {final_train_acc:.1f}% (target: > 95%)")
    checks_passed += 1
else:
    print(f"[FAIL] Train accuracy = {final_train_acc:.1f}% (target: > 95%)")

if 75 < final_test_acc < 96:
    print(f"[PASS] Test accuracy = {final_test_acc:.1f}% (target: 75-96%)")
    checks_passed += 1
else:
    print(f"[FAIL] Test accuracy = {final_test_acc:.1f}% (target: 75-96%)")

if gap > 3:
    print(f"[PASS] Generalization gap = {gap:.1f}% (target: > 3%) -- THIS IS THE PRIVACY LEAK!")
    checks_passed += 1
else:
    print(f"[FAIL] Generalization gap = {gap:.1f}% (target: > 3%)")

print("-" * 50)
print(f"Result: {checks_passed}/{total_checks} checks passed.")
if checks_passed == total_checks:
    print("Phase 1 COMPLETE!")
print("=" * 50)

## Phase 2: Mounting a Membership Inference Attack (~15 min)
**Phase 2 Goal**: 
- Exploit the confidence gap to determine training set membership
- Members should have significantly lower loss than non-members (loss ratio > 1.5x)


**About Membership Inference Attack:**

- **Attack question:** Was a particular example included in the data used to train this model?

- **Attacker's observation:** The attacker examines the model's output for a sample, including its confidence, prediction loss, or output probabilities.

- **Why it works:** Models may behave differently on familiar and unseen data, assigning higher confidence and lower loss to examples memorized during training.

- **Signal used in this lab:** Per-sample loss—lower loss → higher confidence → more likely a member.

- **Privacy risk:** Confirming membership can reveal that a person's record was part of a sensitive medical, financial, or other private dataset, even when the training data itself is never released.

**Note**: For binary classification tasks like PneumoniaMNIST, simple threshold-based AUC may be modest (~0.5) because both groups are mostly correctly classified with high confidence. The *loss ratio* between groups is the more informative metric here.

10: Define the membership signal

- **Goal:** Defines per-sample loss as the confidence signal used by the membership-inference attack.

- **What runs:** `get_membership_signal` runs the model in evaluation mode and is intended to call `F.binary_cross_entropy_with_logits` with `reduction='none'`.

- **Outcome:** The completed function will return one loss value per sample, where lower loss suggests membership.

- **TODO:** Complete TODO 2 by computing the unreduced binary cross-entropy loss for every sample.


In [10]:
def get_membership_signal(model, dataloader):
    """Compute per-sample loss. Lower loss = more confident = more likely a member."""
    signals = []
    model.eval()
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.float().to(device).view(-1, 1)
            outputs = model(images)

            # TODO 2: Compute per-sample loss (1 line)
            # Hint: Use F.binary_cross_entropy_with_logits with reduction='none'
            # (reduction='none' keeps ONE loss per sample instead of averaging)
            per_sample_loss = None  # <-- Replace None

            signals.extend(per_sample_loss.cpu().numpy().flatten())
    return np.array(signals)

print("get_membership_signal function defined.")

11: Score the baseline attack

- **Goal:** Applies the membership signal to member and non-member data and measures attack effectiveness.

- **What runs:** Calls `get_membership_signal`, builds attack labels and negated-loss scores with NumPy, calculates `roc_auc_score`, and compares mean losses through a loss ratio.

- **Outcome:** The cell reports attack AUC, mean member and non-member losses, and an interpretation of privacy leakage.

- **TODO:** Complete TODO 3 by constructing labels and scores, computing the AUC, and calculating the non-member-to-member loss ratio.


In [11]:
member_signals = get_membership_signal(model, member_loader)
nonmember_signals = get_membership_signal(model, nonmember_loader)

# TODO 3: Compute the AUC score and loss ratio for the attack (4 lines)
# Hint: Members are label 1, non-members are label 0
# Hint: Negate loss for scores (lower loss = higher membership confidence)
# Hint: Use roc_auc_score(labels, scores)
# Hint: loss_ratio = mean(nonmember_loss) / mean(member_loss)
attack_labels = None   # <-- np.concatenate of 1s and 0s
attack_scores = None   # <-- np.concatenate of negated signals
mia_auc = None         # <-- roc_auc_score
loss_ratio = None      # <-- ratio of mean losses

print(f"MIA Attack Results:")
print(f"  AUC: {mia_auc:.4f} (0.5 = random guess, 1.0 = perfect attack)")
print(f"  Mean loss - members:     {np.mean(member_signals):.4f}")
print(f"  Mean loss - non-members: {np.mean(nonmember_signals):.4f}")
print(f"  Loss ratio: {loss_ratio:.2f}x (higher = more privacy leakage)")
print(f"")
print(f"Interpretation: Members have {loss_ratio:.1f}x lower loss than non-members.")
print(f"This means the model treats its training data measurably differently!")

12: Plot the baseline attack CDF

- **Goal:** Visualizes how member and non-member per-sample losses differ across their full distributions.

- **What runs:** Defines `empirical_cdf` with `np.sort`, computes both curves, limits extreme values with `np.percentile`, and plots them with Matplotlib.

- **Outcome:** An empirical CDF chart reveals distribution separation, with a larger gap indicating stronger membership leakage.

- **TODO:** N/A


In [12]:
def empirical_cdf(values):
    x = np.sort(values)
    y = np.arange(1, len(x) + 1) / len(x)   # 1/n, 2/n, ..., 1
    return x, y

plt.figure(figsize=(8, 5))
x_m, y_m = empirical_cdf(member_signals)
x_nm, y_nm = empirical_cdf(nonmember_signals)
plt.plot(x_m, y_m, linewidth=2, label='Members (in)')
plt.plot(x_nm, y_nm, linewidth=2, label='Non-members (out)')

x_max = np.percentile(np.concatenate([member_signals, nonmember_signals]), 99)
plt.xlim(0, x_max)
plt.legend(fontsize=12)
plt.xlabel('Per-sample loss  (= -log confidence; lower = more confident)', fontsize=12)
plt.ylabel('Fraction of samples (CDF)', fontsize=12)
plt.title(f'MIA Signal: members are more confident (AUC = {mia_auc:.3f})', fontsize=13)
plt.tight_layout()
plt.show()

13: Validate the baseline attack

- **Goal:** Checks that the computed membership signals exhibit the expected privacy-leakage pattern.

- **What runs:** Compares average losses, tests whether the loss ratio exceeds 1.3, verifies both signal arrays are nonempty, and totals the passing checks.

- **Outcome:** The Phase 2 summary reports whether the membership-inference attack works as expected.

- **TODO:** N/A


In [13]:
print("=" * 50)
print("PHASE 2 VALIDATION")
print("=" * 50)
checks_passed_2 = 0
total_checks_2 = 3

if np.mean(member_signals) < np.mean(nonmember_signals):
    print(f"[PASS] Members have lower avg loss ({np.mean(member_signals):.4f} < {np.mean(nonmember_signals):.4f})")
    checks_passed_2 += 1
else:
    print(f"[FAIL] Expected members to have lower loss than non-members")

if loss_ratio > 1.3:
    print(f"[PASS] Loss ratio = {loss_ratio:.2f}x (target: > 1.3x)")
    checks_passed_2 += 1
else:
    print(f"[FAIL] Loss ratio = {loss_ratio:.2f}x (target: > 1.3x)")

if len(member_signals) > 0 and len(nonmember_signals) > 0:
    print(f"[PASS] Signals computed: {len(member_signals)} members, {len(nonmember_signals)} non-members")
    checks_passed_2 += 1
else:
    print(f"[FAIL] Signals are empty")

print("-" * 50)
print(f"Result: {checks_passed_2}/{total_checks_2} checks passed.")
if checks_passed_2 == total_checks_2:
    print("Phase 2 COMPLETE!")
print("=" * 50)

### Discussion

**Why is the AUC close to 0.5?**

- PneumoniaMNIST is a binary-classification task, and the model is confident on most samples in both groups.
- The per-sample loss is near zero for more than 95% of both members and non-members.
- Because the two groups contain many similarly confident predictions, individual samples are difficult to distinguish.
- An AUC close to 0.5 means this simple attack ranks members only slightly better than random guessing.
- The mean loss difference, represented by the loss ratio, can still reveal an aggregate statistical difference that AUC does not capture clearly.

**Main takeaway:**

- Low per-sample distinguishability does not necessarily mean there is no privacy leakage.
- Members can still receive systematically lower loss than non-members, showing that the model treats its training data differently.
- Across a large population, this small aggregate difference can reveal sensitive membership information.
- More sophisticated attacks, such as shadow-model attacks or LiRA, may amplify this weak signal.

How does the separation in the histogram relate to the generalization gap from Phase 1?

## Phase 3a: Implementing DP-SGD From Scratch (~10 min)

**Phase 3a Goal:** Build an intuitive understanding of DP-SGD by implementing its privacy mechanisms directly.

Standard SGD combines the gradients from a mini-batch and uses them to update the model. DP-SGD modifies this process so that no single training example has too much influence and individual contributions are difficult to recover.

**What we will focus on:**

- **Per-sample gradients:** Compute a separate gradient for each training example instead of immediately averaging the whole batch.
- **Gradient clipping:** Bound each sample's gradient norm so that one patient cannot dominate the model update.
- **Gradient aggregation:** Sum the clipped gradients only after every sample's influence has been limited.
- **Gaussian noise:** Add calibrated random noise to the aggregated gradient to mask individual contributions.
- **Private model updates:** Average the noisy gradient and use it to update the model parameters.

Implementing these steps from scratch makes it easier to see where the privacy protection comes from and how clipping and noise change ordinary training.

**Coming next:** In Phase 3b, we will use the Opacus library to automate per-sample clipping, noise addition, and privacy accounting. We will then compare the manual and library-based approaches.

> **Note:** This educational implementation uses an explicit per-sample Python loop, so it may run slowly. It is designed to clarify the mechanism rather than optimize performance.

14: Implement DP-SGD from scratch

- **Goal:** Demonstrates the mechanics of private training by manually clipping per-sample gradients and adding Gaussian noise.

- **What runs:** Uses `get_noise_multiplier`, computes individual gradients in `compute_dp_gradients`, accumulates noisy gradients, and trains a fresh model with manual SGD updates.

- **Outcome:** A manually private model is trained and evaluated so its privacy and utility can be compared with Opacus.

- **TODO:** Complete TODO 4a by clipping each sample's gradients, and TODO 4b by drawing Gaussian noise with the specified standard deviation.


In [19]:
from opacus.accountants.utils import get_noise_multiplier

# --- Hyper-parameters for the manual run ---------------------------------
MANUAL_EPOCHS = 15            # per-sample loop is slow, so keep this modest
MANUAL_MAX_GRAD_NORM = 1.0    # C: the clipping bound (limits each sample's influence)
MANUAL_EPSILON = 8.0          # target privacy budget for this demo
MANUAL_DELTA = 1e-5
manual_sample_rate = 64 / len(member_dataset)  # batch_size / dataset_size

# Ask Opacus's accountant ONLY for the required noise scale (not for training).
manual_noise_multiplier = get_noise_multiplier(
    target_epsilon=MANUAL_EPSILON,
    target_delta=MANUAL_DELTA,
    sample_rate=manual_sample_rate,
    epochs=MANUAL_EPOCHS,
)
print(f"Computed noise multiplier (sigma): {manual_noise_multiplier:.4f}")


def compute_dp_gradients(model, images, labels, criterion, max_grad_norm, noise_multiplier):
    """Return the DP gradient for one mini-batch: per-sample clip, sum, add noise, average."""
    accumulated = {name: torch.zeros_like(p) for name, p in model.named_parameters()}
    batch_size = images.size(0)

    # --- Process the batch ONE sample at a time so we can clip each individually ---
    for i in range(batch_size):
        model.zero_grad()
        output = model(images[i:i + 1])          # forward on a single example
        loss = criterion(output, labels[i:i + 1])
        loss.backward()                          # gradient of THIS sample only

        # ============================================================
        # TODO 4a: CLIP this single sample's gradient so its L2 norm <= max_grad_norm.
        #          This bounds how much one patient can influence the model (sensitivity).
        # Hint: torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        # ============================================================
        pass  # <-- Replace this line with the clipping call

        for name, param in model.named_parameters():
            accumulated[name] += param.grad      # add the clipped gradient to the sum

    for name, param in model.named_parameters():
        # ============================================================
        # TODO 4b: DRAW Gaussian noise with mean 0 and std = noise_multiplier * max_grad_norm.
        #          This masks individual contributions and gives the formal guarantee.
        # Hint: torch.normal(0.0, noise_multiplier * max_grad_norm,
        #                    size=param.shape, device=param.device)
        # ============================================================
        noise = None  # <-- Replace None with the noise tensor
        accumulated[name] = (accumulated[name] + noise) / batch_size
    return accumulated


# --- Train a fresh model with our hand-written DP gradients ---------------
manual_loader = DataLoader(member_dataset, batch_size=64, shuffle=True)
model_manual = create_model()                    # same architecture as every other phase
optimizer_manual = torch.optim.SGD(model_manual.parameters(), lr=0.2)
criterion_manual = nn.BCEWithLogitsLoss()

print(f"Training with manual DP-SGD for {MANUAL_EPOCHS} epochs (slower -- per-sample loop)...")
for epoch in range(MANUAL_EPOCHS):
    model_manual.train()
    for images, labels in manual_loader:
        images = images.to(device)
        labels = labels.float().to(device).view(-1, 1)
        grads = compute_dp_gradients(model_manual, images, labels, criterion_manual,
                                     MANUAL_MAX_GRAD_NORM, manual_noise_multiplier)
        for name, param in model_manual.named_parameters():
            param.grad = grads[name]             # inject our DP gradient
        optimizer_manual.step()                  # ordinary SGD update using it

manual_test_acc = evaluate_accuracy(model_manual, test_loader)
print(f"Manual DP-SGD test accuracy: {manual_test_acc:.1f}%")


## Phase 3b: DP-SGD with Opacus (~15 min)

**Phase 3b Goal:** Apply the DP-SGD ideas from Phase 3a using a library and observe whether membership-inference leakage decreases.

**Target:** DP model MIA AUC < 0.58

**What is Opacus?**

[Opacus](https://opacus.ai/) is an open-source library for training PyTorch models with differential privacy. Its `PrivacyEngine` wraps familiar PyTorch training components so we can use DP-SGD without manually implementing every privacy operation.

**What will Opacus handle for us?**

- Compute and clip per-sample gradients.
- Add calibrated Gaussian noise during optimization.
- Track the privacy budget spent during training.
- Preserve a training loop that closely resembles standard PyTorch code.

**Connection to Phase 3a:** We implemented clipping and noise ourselves to understand the mechanism. We will now use Opacus to perform those operations automatically and then compare the privacy and utility results.

15: Configure private training with Opacus

- **Goal:** Prepares a fresh model and optimizer for differentially private stochastic gradient descent.

- **What runs:** Sets the privacy hyperparameters, creates an SGD optimizer and member-data loader, and is intended to wrap them using `PrivacyEngine.make_private_with_epsilon`.

- **Outcome:** The private model, optimizer, and loader will be configured for the target epsilon and delta, and the resulting noise multiplier will be printed.

- **TODO:** Complete TODO 5 by calling `make_private_with_epsilon` with the model, optimizer, loader, epochs, privacy targets, and clipping norm.


In [ ]:
EPOCHS_DP = 20
EPSILON = 2.0
DELTA = 1e-5
MAX_GRAD_NORM = 1.0
LR_DP = 0.1

# Fresh model + plain SGD (DP-SGD builds on SGD, not Adam).
model_dp = create_model()
optimizer_dp = torch.optim.SGD(model_dp.parameters(), lr=LR_DP)

# Opacus wraps the loader to enable per-sample gradient computation.
train_loader_dp = DataLoader(member_dataset, batch_size=64, shuffle=True)


# TODO 5: Attach PrivacyEngine to make training differentially private
# Hint:
#   1. Create a PrivacyEngine to manage the private training setup and privacy accounting.
#   2. Call make_private_with_epsilon(...) and assign its three returned objects back to
#      model_dp, optimizer_dp, and train_loader_dp.
#   3. Match each keyword argument below with the model, optimizer, loader, and privacy
#      hyperparameter variables already defined at the top of this cell.
privacy_engine = PrivacyEngine()
model_dp, optimizer_dp, train_loader_dp = privacy_engine.make_private_with_epsilon(
    module=,
    optimizer=,
    data_loader=,
    epochs=,
    target_epsilon=,
    target_delta=,
    max_grad_norm=,
)

print(f"Using noise_multiplier = {optimizer_dp.noise_multiplier:.4f}")
print(f"Target: epsilon = {EPSILON}, delta = {DELTA}")

16: Train the Opacus DP model

- **Goal:** Trains the differentially private model while tracking accuracy and privacy-budget consumption.

- **What runs:** Runs forward and backward passes with `BCEWithLogitsLoss`; the Opacus-wrapped optimizer clips gradients and adds noise during `step`; `get_epsilon` tracks privacy spending.

- **Outcome:** Accuracy and epsilon are reported every five epochs, followed by the final DP test accuracy.

- **TODO:** Complete TODO 6 by implementing the private training step from `optimizer_dp.zero_grad()` through `optimizer_dp.step()`.


In [ ]:
criterion_dp = nn.BCEWithLogitsLoss()
train_accs_dp = []
test_accs_dp = []

print(f"Training DP model for {EPOCHS_DP} epochs...")
for epoch in range(EPOCHS_DP):
    model_dp.train()
    for images, labels in train_loader_dp:
        images = images.to(device)
        labels = labels.float().to(device).view(-1, 1)
        
        # TODO 6: Complete the DP training step (5 lines)
        # Hint: Clear the old gradients, run the forward pass, compute the loss,
        # backpropagate, and take an optimizer step. Opacus applies clipping and
        # noise automatically when optimizer_dp.step() is called.

        optimizer_dp.zero_grad()
        outputs = None  # <-- Replace None with forward pass
        loss = None     # <-- Replace None with loss computation
        # <-- Add backward and step calls

    if (epoch + 1) % 5 == 0:
        train_acc = evaluate_accuracy(model_dp, train_loader_dp)
        test_acc_dp_curr = evaluate_accuracy(model_dp, test_loader)
        train_accs_dp.append(train_acc)
        test_accs_dp.append(test_acc_dp_curr)
        # Track the privacy budget actually spent so far.
        eps_spent = privacy_engine.get_epsilon(DELTA)
        print(f"Epoch {epoch+1}: Train={train_acc:.1f}%, Test={test_acc_dp_curr:.1f}%, eps={eps_spent:.2f}")

test_acc_dp = evaluate_accuracy(model_dp, test_loader)
print(f"\nFinal DP model test accuracy: {test_acc_dp:.1f}%")

17: Attack and compare the DP model

- **Goal:** Repeats the membership-inference attack on the private model and compares it with the baseline.

- **What runs:** Unwraps the Opacus model when needed, obtains member and non-member signals, constructs labels and scores, computes AUC and loss ratio, and formats a comparison table.

- **Outcome:** The table shows whether differential privacy reduces the distinguishability of members and non-members.

- **TODO:** Complete TODO 7 by computing DP signals, attack labels and scores, AUC, and loss ratio using the Phase 2 procedure.


In [16]:
model_dp_eval = model_dp._module if hasattr(model_dp, '_module') else model_dp

# TODO 7: Re-run the membership-inference attack on the DP model
member_signals_dp = None      # <-- Get signals for members
nonmember_signals_dp = None   # <-- Get signals for non-members
attack_labels_dp = None       # <-- Concatenate labels (same as Phase 2)
attack_scores_dp = None       # <-- Concatenate negated signals
mia_auc_dp = None             # <-- Compute AUC
loss_ratio_dp = None          # <-- Compute loss ratio

print(f"{'Metric':<25} {'Baseline':>10} {'DP Model':>10} {'Change':>10}")
print("-" * 55)
print(f"{'MIA AUC':<25} {mia_auc:>10.4f} {mia_auc_dp:>10.4f} {mia_auc_dp - mia_auc:>+10.4f}")
print(f"{'Loss Ratio':<25} {loss_ratio:>10.2f}x {loss_ratio_dp:>10.2f}x {'improved' if loss_ratio_dp < loss_ratio else '':>10}")
print(f"{'Mean Loss (members)':<25} {np.mean(member_signals):>10.4f} {np.mean(member_signals_dp):>10.4f}")
print(f"{'Mean Loss (non-members)':<25} {np.mean(nonmember_signals):>10.4f} {np.mean(nonmember_signals_dp):>10.4f}")
print()
print(f"DP makes members LESS distinguishable from non-members!")

18: Compare baseline and DP loss distributions

- **Goal:** Places the baseline and private membership-signal distributions side by side.

- **What runs:** Uses `empirical_cdf`, `np.percentile`, and a two-panel Matplotlib figure to plot member and non-member loss curves for both models.

- **Outcome:** The resulting charts visually show whether DP narrows the gap between member and non-member distributions.

- **TODO:** N/A


In [17]:
def empirical_cdf(values):
    x = np.sort(values)
    y = np.arange(1, len(x) + 1) / len(x)
    return x, y

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Baseline model ---
x_m, y_m = empirical_cdf(member_signals)
x_nm, y_nm = empirical_cdf(nonmember_signals)
ax1.plot(x_m, y_m, linewidth=2, label='Members (in)')
ax1.plot(x_nm, y_nm, linewidth=2, label='Non-members (out)')
ax1.set_xlim(0, np.percentile(np.concatenate([member_signals, nonmember_signals]), 99))
ax1.set_xlabel('Per-sample loss (-log confidence)', fontsize=11)
ax1.set_ylabel('Fraction of samples (CDF)', fontsize=11)
ax1.set_title(f'Baseline Model (AUC = {mia_auc:.3f})', fontsize=13)
ax1.legend(fontsize=11)

# --- DP model ---
x_m_dp, y_m_dp = empirical_cdf(member_signals_dp)
x_nm_dp, y_nm_dp = empirical_cdf(nonmember_signals_dp)
ax2.plot(x_m_dp, y_m_dp, linewidth=2, label='Members (in)')
ax2.plot(x_nm_dp, y_nm_dp, linewidth=2, label='Non-members (out)')
ax2.set_xlim(0, np.percentile(np.concatenate([member_signals_dp, nonmember_signals_dp]), 99))
ax2.set_xlabel('Per-sample loss (-log confidence)', fontsize=11)
ax2.set_ylabel('Fraction of samples (CDF)', fontsize=11)
ax2.set_title(f'DP Model (AUC = {mia_auc_dp:.3f})', fontsize=13)
ax2.legend(fontsize=11)

plt.suptitle('Membership Inference: gap between curves = privacy leakage', fontsize=14)
plt.tight_layout()
plt.show()

19: Validate differential privacy results

- **Goal:** Checks whether DP reduces membership leakage while retaining useful accuracy and respecting the privacy budget.

- **What runs:** Compares baseline and DP loss ratios and accuracy gaps, evaluates DP test accuracy, queries `privacy_engine.get_epsilon`, and counts four validations.

- **Outcome:** The Phase 3 summary reports whether leakage decreased, utility remained acceptable, and epsilon stayed within budget.

- **TODO:** N/A


In [18]:
print("=" * 50)
print("PHASE 3 VALIDATION")
print("=" * 50)
checks_passed_3 = 0
total_checks_3 = 4

if loss_ratio_dp < loss_ratio:
    print(f"[PASS] Loss ratio reduced: {loss_ratio:.2f}x -> {loss_ratio_dp:.2f}x")
    checks_passed_3 += 1
else:
    print(f"[FAIL] Loss ratio not reduced: {loss_ratio:.2f}x -> {loss_ratio_dp:.2f}x")

test_acc_dp = evaluate_accuracy(model_dp_eval, test_loader)
if test_acc_dp > 55:
    print(f"[PASS] DP model test accuracy = {test_acc_dp:.1f}% (target: > 55%)")
    checks_passed_3 += 1
else:
    print(f"[FAIL] DP model test accuracy = {test_acc_dp:.1f}% (target: > 55%)")

dp_member_acc = evaluate_accuracy(model_dp_eval, member_loader)
dp_nonmem_acc = evaluate_accuracy(model_dp_eval, nonmember_loader)
dp_acc_gap = dp_member_acc - dp_nonmem_acc
baseline_acc_gap = evaluate_accuracy(model, member_loader) - evaluate_accuracy(model, nonmember_loader)
if abs(dp_acc_gap) < abs(baseline_acc_gap):
    print(f"[PASS] Accuracy gap reduced: {baseline_acc_gap:.1f}% -> {dp_acc_gap:.1f}%")
    checks_passed_3 += 1
else:
    print(f"[FAIL] Accuracy gap not reduced: {baseline_acc_gap:.1f}% -> {dp_acc_gap:.1f}%")

eps_final = privacy_engine.get_epsilon(DELTA)
if eps_final <= EPSILON * 1.1:
    print(f"[PASS] Epsilon spent = {eps_final:.2f} (budget: {EPSILON})")
    checks_passed_3 += 1
else:
    print(f"[FAIL] Epsilon spent = {eps_final:.2f} (budget: {EPSILON})")

print("-" * 50)
print(f"Result: {checks_passed_3}/{total_checks_3} checks passed.")
if checks_passed_3 == total_checks_3:
    print("Phase 3 COMPLETE!")
print("=" * 50)

### Discussion

Notice how DP-SGD reduces the loss ratio (members become less distinguishable from non-members). The gradient clipping bounds how much any single sample can influence the model, and noise injection further hides individual contributions.

**Key observation**: The loss ratio dropping from ~1.8x to ~0.8-1.0x means the model no longer "remembers" its training data differently from unseen data. This is the formal privacy guarantee at work!

Why do the member/non-member distributions overlap more in the DP model?

20: Compare manual DP with Opacus

- **Goal:** Compares privacy leakage and test accuracy across the baseline, Opacus DP-SGD, and manual DP-SGD models.

- **What runs:** Calls `get_membership_signal`, calculates the manual model's loss ratio with NumPy, and prints a formatted three-model comparison.

- **Outcome:** The output shows whether the hand-written clipping-and-noise procedure reproduces Opacus's privacy protection.

- **TODO:** N/A


In [20]:
member_signals_manual = get_membership_signal(model_manual, member_loader)
nonmember_signals_manual = get_membership_signal(model_manual, nonmember_loader)
loss_ratio_manual = np.mean(nonmember_signals_manual) / (np.mean(member_signals_manual) + 1e-8)

print(f"{'Approach':<30}{'Loss Ratio':>12}{'Test Acc':>11}")
print("-" * 53)
print(f"{'Baseline (no DP)':<30}{loss_ratio:>11.2f}x{test_accs[-1]:>10.1f}%")
print(f"{'Opacus DP-SGD':<30}{loss_ratio_dp:>11.2f}x{test_acc_dp:>10.1f}%")
print(f"{'Manual DP-SGD (from scratch)':<30}{loss_ratio_manual:>11.2f}x{manual_test_acc:>10.1f}%")
print()
print("Both DP methods shrink the loss ratio toward 1.0 -- our hand-written")
print("clip + noise reproduces the privacy protection that Opacus provides.")


### Discussion: Manual vs Opacus

You just built DP-SGD from first principles! Key takeaways:

- The **clip** step (`clip_grad_norm_`) is what bounds each sample's influence — it caps the *sensitivity*.
- The **noise** step (`torch.normal(...)`) is what provides the formal (epsilon, delta) guarantee.
- Opacus automates exactly this, but computes per-sample gradients *vectorized* (no Python loop), which is why it
  is dramatically faster while implementing the same math you just wrote.

**Question to ponder**: Our manual version loops over samples one at a time to get per-sample gradients. Why is that
slow, and how might a library compute per-sample gradients for a whole batch at once (hint: look up *functorch* /
`torch.func.vmap`)?

## Phase 4: The Privacy-Utility Trade-off (~12 min)
**Goal**: Visualize how privacy and utility trade off across different epsilon values
**Target**: Plot shows clear trend — stronger privacy (lower epsilon) costs utility

In practice, there's no free lunch. We evaluate pre-trained checkpoints at various epsilon levels.

21: Evaluate multiple privacy budgets

- **Goal:** Measures the privacy-utility trade-off across several epsilon values using saved model checkpoints.

- **What runs:** Defines `load_checkpoint`, loads each model with `torch.load` or uses a fallback, and is intended to call `evaluate_accuracy`, `get_membership_signal`, and `roc_auc_score` for every epsilon.

- **Outcome:** A results list and table will contain test accuracy, attack AUC, and loss ratio for six privacy budgets.

- **TODO:** Complete TODO 8 by calculating accuracy, membership signals, labels, scores, AUC, and loss ratio for every checkpoint.


In [ ]:
import os
MODEL_TYPE = 'FC'

def load_checkpoint(epsilon):
    """Load pre-trained model checkpoint for given epsilon."""
    if epsilon == float('inf'):
        fname = f"checkpoints/{MODEL_TYPE.lower()}_eps_inf.pt"   # inf = no privacy
    else:
        fname = f"checkpoints/{MODEL_TYPE.lower()}_eps_{epsilon}.pt"

    model_ckpt = create_model()
    if os.path.exists(fname):
        model_ckpt.load_state_dict(torch.load(fname, map_location=device))
    else:
        # Fallback so the notebook still runs if checkpoints are missing.
        print(f"WARNING: {fname} not found. Using current models instead.")
        if epsilon == float('inf'):
            model_ckpt.load_state_dict(model.state_dict())
        else:
            state = model_dp._module.state_dict() if hasattr(model_dp, '_module') else model_dp.state_dict()
            model_ckpt.load_state_dict(state)
    return model_ckpt

# From strong privacy (0.5) to none (inf).
epsilons = [0.5, 1.0, 2.0, 5.0, 10.0, float('inf')]
results = []

print("Evaluating models at different epsilon levels...")
print("-" * 60)
print(f"{'Epsilon':>8} {'Test Acc':>10} {'Loss Ratio':>12} {'MIA AUC':>10}")
print("-" * 60)
for eps in epsilons:
    model_ckpt = load_checkpoint(eps)

    # TODO 8: For each checkpoint, compute test accuracy, loss ratio, and MIA AUC (6 lines)
    # Hint: Use evaluate_accuracy() for test accuracy
    # Hint: Use get_membership_signal() for member and non-member signals
    # Hint: Compute AUC and loss_ratio the same way as Phase 2
    test_acc_ckpt = None   # <-- evaluate_accuracy
    signals_m = None       # <-- get_membership_signal on member_loader
    signals_nm = None      # <-- get_membership_signal on nonmember_loader
    labels_ckpt = None     # <-- concatenate labels
    scores_ckpt = None     # <-- concatenate negated signals
    auc_ckpt = None        # <-- roc_auc_score
    lr_ckpt = None         # <-- loss ratio

    results.append({'epsilon': eps, 'test_acc': test_acc_ckpt, 'mia_auc': auc_ckpt, 'loss_ratio': lr_ckpt})
    eps_str = "inf" if eps == float('inf') else f"{eps}"
    print(f"{eps_str:>8} {test_acc_ckpt:>9.1f}% {lr_ckpt:>11.2f}x {auc_ckpt:>10.4f}")

22: Plot the privacy-utility trade-off

- **Goal:** Visualizes how model utility and membership leakage change as the privacy budget becomes weaker.

- **What runs:** Extracts accuracy and loss-ratio values from `results` and uses a two-panel Matplotlib figure with categorical epsilon labels.

- **Outcome:** The plots show accuracy versus epsilon and leakage versus epsilon, with a reference line marking a loss ratio of one.

- **TODO:** N/A


In [22]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: utility (accuracy) as the privacy budget loosens.
eps_labels = ['0.5', '1.0', '2.0', '5.0', '10.0', 'inf']
accs = [r['test_acc'] for r in results]
ax1.plot(range(len(results)), accs, 'bo-', markersize=8)
ax1.set_xticks(range(len(results)))
ax1.set_xticklabels(eps_labels)
ax1.set_xlabel('Epsilon (privacy budget)', fontsize=12)
ax1.set_ylabel('Test Accuracy (%)', fontsize=12)
ax1.set_title('Utility vs Privacy Budget', fontsize=14)
ax1.grid(True, alpha=0.3)

# Plot 2: leakage (loss ratio) as the privacy budget loosens.
lrs = [r['loss_ratio'] for r in results]
ax2.plot(range(len(results)), lrs, 'ro-', markersize=8)
ax2.set_xticks(range(len(results)))
ax2.set_xticklabels(eps_labels)
ax2.set_xlabel('Epsilon (privacy budget)', fontsize=12)
ax2.set_ylabel('Loss Ratio (member vs non-member)', fontsize=12)
ax2.set_title('Privacy Leakage vs Privacy Budget', fontsize=14)
ax2.axhline(y=1.0, color='gray', linestyle='--', alpha=0.7, label='No leakage (ratio=1)')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.suptitle('The Privacy-Utility Trade-off', fontsize=15)
plt.tight_layout()
plt.show()

23: Validate the privacy-budget sweep

- **Goal:** Checks that all epsilon settings were evaluated and that the expected utility trade-off is visible.

- **What runs:** Verifies the result count, compares strongest-privacy and no-privacy accuracy, computes the accuracy range, and reports three checks.

- **Outcome:** The Phase 4 summary confirms whether the sweep contains six points and demonstrates improving utility as epsilon increases.

- **TODO:** N/A


In [23]:
print("=" * 50)
print("PHASE 4 VALIDATION")
print("=" * 50)
checks_passed_4 = 0
total_checks_4 = 3

if len(results) == 6:
    print(f"[PASS] Evaluated {len(results)} epsilon values")
    checks_passed_4 += 1
else:
    print(f"[FAIL] Expected 6 results, got {len(results)}")

# Utility should generally improve as the privacy budget loosens.
accs = [r['test_acc'] for r in results]
if accs[-1] > accs[0]:
    print(f"[PASS] Higher epsilon -> better utility: eps=0.5 acc={accs[0]:.1f}%, eps=inf acc={accs[-1]:.1f}%")
    checks_passed_4 += 1
else:
    print(f"[FAIL] Expected higher epsilon to give better accuracy")

# The gap between strongest and weakest privacy should be visible.
acc_range = max(accs) - min(accs)
if acc_range > 1.0:
    print(f"[PASS] Accuracy range = {acc_range:.1f}% (trade-off is visible)")
    checks_passed_4 += 1
else:
    print(f"[FAIL] Accuracy range = {acc_range:.1f}% (trade-off not visible enough)")

print("-" * 50)
print(f"Result: {checks_passed_4}/{total_checks_4} checks passed.")
if checks_passed_4 == total_checks_4:
    print("Phase 4 COMPLETE!")
print("=" * 50)

### Final Discussion

Imagine that this model will support pneumonia diagnosis using sensitive patient data.

**1. Which epsilon would you choose?**

- **Hint:** Start by comparing the accuracy and membership-leakage plots. A smaller epsilon generally provides stronger privacy, but the additional noise may reduce model utility.
- **Possible explanation:** A cautious team might select a small epsilon, such as 0.5, because medical-data membership is sensitive. Another team might justify a moderate value, such as 2.0 or 5.0, if it provides a meaningful accuracy improvement while keeping leakage acceptably low. The choice should be supported by evidence from the trade-off results rather than accuracy alone.

**2. What factors should influence this decision?**

- **Hint:** Consider both the consequences of a privacy failure and the consequences of an incorrect diagnosis.
- **Possible considerations:**
  - The sensitivity of the patient data and the harm caused by revealing membership.
  - The minimum accuracy required for the model's intended clinical role.
  - The observed attack AUC, loss ratio, and utility at each epsilon.
  - The dataset size, class balance, and whether some patient groups experience a larger accuracy loss.
  - The attacker's knowledge and access to model outputs.
  - The number of training runs or releases, because privacy loss can accumulate across repeated uses of the data.
  - Relevant institutional policies, legal requirements, and input from clinicians and patients.

**3. Is there a single correct epsilon?**

- **Hint:** Epsilon is a policy and risk-management choice as well as a technical parameter.
- **Possible explanation:** There is usually no universally correct value. Two deployments can reasonably choose different privacy budgets because their data sensitivity, threat models, accuracy requirements, and acceptable risks differ. The important requirement is to justify the choice, document its assumptions, and verify the resulting privacy and utility empirically.

Congratulations! You've completed the full attack-defense-tradeoff cycle of privacy in ML.